In [11]:
#Importar Bibliotecas
import pandas as pd

In [12]:
#Importar Dados
from pathlib import Path

raiz_projeto = Path.cwd()
if raiz_projeto.name == 'notebooks':
    raiz_projeto = raiz_projeto.parent

df = pd.read_csv(raiz_projeto / 'data' / 'raw' / '1.dirty_cafe_sales.csv')

In [13]:
df.shape

(10000, 8)

In [14]:
df.columns

Index(['Transaction ID', 'Item', 'Quantity', 'Price Per Unit', 'Total Spent',
       'Payment Method', 'Location', 'Transaction Date'],
      dtype='str')

In [15]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Transaction ID    10000 non-null  str  
 1   Item              9667 non-null   str  
 2   Quantity          9862 non-null   str  
 3   Price Per Unit    9821 non-null   str  
 4   Total Spent       9827 non-null   str  
 5   Payment Method    7421 non-null   str  
 6   Location          6735 non-null   str  
 7   Transaction Date  9841 non-null   str  
dtypes: str(8)
memory usage: 625.1 KB


In [16]:
df.isnull().sum()

Transaction ID         0
Item                 333
Quantity             138
Price Per Unit       179
Total Spent          173
Payment Method      2579
Location            3265
Transaction Date     159
dtype: int64

In [17]:
df.duplicated().sum()

np.int64(0)

In [18]:
# Definir Primary Key
df['Transaction ID'].value_counts()

Transaction ID
TXN_1961373    1
TXN_4977031    1
TXN_4271903    1
TXN_7034554    1
TXN_3160411    1
              ..
TXN_7672686    1
TXN_9659401    1
TXN_5255387    1
TXN_7695629    1
TXN_6170729    1
Name: count, Length: 10000, dtype: int64

In [19]:
# Preservar a base original
df_limpo = df.copy()

colunas_numericas = ['Quantity', 'Price Per Unit', 'Total Spent']

for coluna in colunas_numericas:
    df_limpo[coluna] = pd.to_numeric(
        df_limpo[coluna],
        errors='coerce'
    )

In [20]:
df_limpo[colunas_numericas].isna().sum()

Quantity          479
Price Per Unit    533
Total Spent       502
dtype: int64

In [21]:
import numpy as np

# Selecionar linhas com os três valores presentes
completas = df_limpo.dropna(subset=colunas_numericas).copy()

# Comparar o total registrado com quantidade × preço
completas['total_coerente'] = np.isclose(
    completas['Quantity'] * completas['Price Per Unit'],
    completas['Total Spent'],
    rtol=0,
    atol=0.01
)

completas['total_coerente'].value_counts()

total_coerente
True    8544
Name: count, dtype: int64

In [22]:
candidatas = (
    df_limpo['Quantity'].isna()
    & df_limpo['Total Spent'].notna()
    & (df_limpo['Price Per Unit'] > 0)
)

quantidade_calculada = (
    df_limpo.loc[candidatas, 'Total Spent']
    / df_limpo.loc[candidatas, 'Price Per Unit']
)

# Aceitar apenas resultados finitos, positivos e próximos de um inteiro
validas = (
    np.isfinite(quantidade_calculada)
    & (quantidade_calculada > 0)
    & np.isclose(
        quantidade_calculada,
        quantidade_calculada.round(),
        rtol=0,
        atol=1e-9
    )
)

indices = quantidade_calculada.index[validas]

df_limpo.loc[indices, 'Quantity'] = (
    quantidade_calculada.loc[indices].round()
)

print('Quantidades recuperadas:', len(indices))
print('Quantidades ainda ausentes:', df_limpo['Quantity'].isna().sum())

Quantidades recuperadas: 441
Quantidades ainda ausentes: 38


In [23]:
candidatas = (
    df_limpo['Price Per Unit'].isna()
    & df_limpo['Total Spent'].notna()
    & (df_limpo['Quantity'] > 0)
)

preco_calculado = (
    df_limpo.loc[candidatas, 'Total Spent']
    / df_limpo.loc[candidatas, 'Quantity']
)

validas = (
    np.isfinite(preco_calculado)
    & (preco_calculado > 0)
)

indices = preco_calculado.index[validas]

df_limpo.loc[indices, 'Price Per Unit'] = preco_calculado.loc[indices]

print('Preços recuperados:', len(indices))
print('Preços ainda ausentes:', df_limpo['Price Per Unit'].isna().sum())

Preços recuperados: 495
Preços ainda ausentes: 38


In [24]:
candidatas = (
    df_limpo['Total Spent'].isna()
    & (df_limpo['Quantity'] > 0)
    & (df_limpo['Price Per Unit'] > 0)
)

total_calculado = (
    df_limpo.loc[candidatas, 'Quantity']
    * df_limpo.loc[candidatas, 'Price Per Unit']
)

validas = np.isfinite(total_calculado)

indices = total_calculado.index[validas]

df_limpo.loc[indices, 'Total Spent'] = total_calculado.loc[indices]

print('Totais recuperados:', len(indices))
print('Totais ainda ausentes:', df_limpo['Total Spent'].isna().sum())

Totais recuperados: 462
Totais ainda ausentes: 40


In [25]:
df_limpo[colunas_numericas].isna().sum()

Quantity          38
Price Per Unit    38
Total Spent       40
dtype: int64

In [26]:
referencia = df[['Item', 'Price Per Unit']].copy()

referencia['Price Per Unit'] = pd.to_numeric(
    referencia['Price Per Unit'],
    errors='coerce'
)

referencia = referencia.loc[
    referencia['Item'].notna()
    & ~referencia['Item'].isin(['UNKNOWN', 'ERROR'])
    & (referencia['Price Per Unit'] > 0)
]

resumo_precos = referencia.groupby('Item')['Price Per Unit'].agg(
    precos_distintos='nunique',
    menor_preco='min',
    maior_preco='max',
    observacoes='count'
)

resumo_precos

,precos_distintos,menor_preco,maior_preco,observacoes
Item,,,,
Cake,1,3.0,3.0,1085
Coffee,1,2.0,2.0,1108
Cookie,1,1.0,1.0,1026
Juice,1,3.0,3.0,1110
Salad,1,5.0,5.0,1082
Sandwich,1,4.0,4.0,1082
Smoothie,1,4.0,4.0,1036
Tea,1,1.5,1.5,1023


In [27]:
# Usar somente produtos com um único preço observado
mapa_precos = resumo_precos.loc[
    resumo_precos['precos_distintos'] == 1,
    'menor_preco'
]

precos_ausentes_antes = df_limpo['Price Per Unit'].isna().sum()

df_limpo['Price Per Unit'] = df_limpo['Price Per Unit'].fillna(
    df_limpo['Item'].map(mapa_precos)
)

print(
    'Preços recuperados por produto:',
    precos_ausentes_antes - df_limpo['Price Per Unit'].isna().sum()
)
print('Preços ainda ausentes:', df_limpo['Price Per Unit'].isna().sum())

Preços recuperados por produto: 32
Preços ainda ausentes: 6


In [28]:
# Recuperar quantidades usando os preços preenchidos por produto
candidatas = (
    df_limpo['Quantity'].isna()
    & df_limpo['Total Spent'].notna()
    & (df_limpo['Price Per Unit'] > 0)
)

quantidade_calculada = (
    df_limpo.loc[candidatas, 'Total Spent']
    / df_limpo.loc[candidatas, 'Price Per Unit']
)

validas = (
    np.isfinite(quantidade_calculada)
    & (quantidade_calculada > 0)
    & np.isclose(
        quantidade_calculada,
        quantidade_calculada.round(),
        rtol=0,
        atol=1e-9
    )
)

indices = quantidade_calculada.index[validas]

df_limpo.loc[indices, 'Quantity'] = (
    quantidade_calculada.loc[indices].round()
)

print('Quantidades recuperadas nesta etapa:', len(indices))


# Recuperar totais usando quantidade e preço disponíveis
candidatas = (
    df_limpo['Total Spent'].isna()
    & (df_limpo['Quantity'] > 0)
    & (df_limpo['Price Per Unit'] > 0)
)

total_calculado = (
    df_limpo.loc[candidatas, 'Quantity']
    * df_limpo.loc[candidatas, 'Price Per Unit']
)

validas = np.isfinite(total_calculado)
indices = total_calculado.index[validas]

df_limpo.loc[indices, 'Total Spent'] = total_calculado.loc[indices]

print('Totais recuperados nesta etapa:', len(indices))

Quantidades recuperadas nesta etapa: 15
Totais recuperados nesta etapa: 17


In [29]:
df_limpo[colunas_numericas].isna().sum()

Quantity          23
Price Per Unit     6
Total Spent       23
dtype: int64

In [30]:
pendentes = df_limpo.loc[
    df_limpo[colunas_numericas].isna().any(axis=1),
    ['Transaction ID', 'Item'] + colunas_numericas
]

pendentes

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent
236,TXN_8562645,Salad,NaN,5.0,NaN
278,TXN_3229409,Juice,NaN,3.0,NaN
641,TXN_2962976,Juice,NaN,3.0,NaN
738,TXN_8696094,Sandwich,NaN,4.0,NaN
1761,TXN_3611851,NaN,4.0,NaN,NaN
2289,TXN_7524977,UNKNOWN,4.0,NaN,NaN
2796,TXN_9188692,Cake,NaN,3.0,NaN
3203,TXN_4565754,Smoothie,NaN,4.0,NaN
3224,TXN_6297232,Coffee,NaN,2.0,NaN
3401,TXN_3251829,Tea,NaN,1.5,NaN


In [31]:
total_transacoes = len(df_limpo)

resumo_ausencias = (
    df_limpo[colunas_numericas]
    .isna()
    .sum()
    .to_frame(name='transacoes_ausentes')
)

resumo_ausencias['percentual_do_total'] = (
    resumo_ausencias['transacoes_ausentes'] / total_transacoes * 100
).round(2)

resumo_ausencias

,transacoes_ausentes,percentual_do_total
Quantity,23,0.23
Price Per Unit,6,0.06
Total Spent,23,0.23


In [32]:
transacoes_incompletas = (
    df_limpo[colunas_numericas].isna().any(axis=1).sum()
)

percentual_incompletas = transacoes_incompletas / total_transacoes * 100

print(
    f'{transacoes_incompletas} de {total_transacoes} transações '
    f'({percentual_incompletas:.2f}%) têm dados numéricos incompletos.'
)

26 de 10000 transações (0.26%) têm dados numéricos incompletos.


In [33]:
df_limpo['Payment Method'].value_counts(dropna=False)

Payment Method
NaN               2579
Digital Wallet    2291
Credit Card       2273
Cash              2258
ERROR              306
UNKNOWN            293
Name: count, dtype: int64

In [34]:
df_limpo['Location'].value_counts(dropna=False)

Location
NaN         3265
Takeaway    3022
In-store    3017
ERROR        358
UNKNOWN      338
Name: count, dtype: int64

In [35]:
colunas_categoricas = ['Payment Method', 'Location']

for coluna in colunas_categoricas:
    df_limpo[coluna] = (
        df_limpo[coluna]
        .str.strip()
        .replace(['UNKNOWN', 'ERROR', ''], pd.NA)
        .fillna('Não informado')
    )

In [36]:
for coluna in colunas_categoricas:
    print(f'\n{coluna}')
    print(df_limpo[coluna].value_counts(dropna=False))


Payment Method
Payment Method
Não informado     3178
Digital Wallet    2291
Credit Card       2273
Cash              2258
Name: count, dtype: int64

Location
Location
Não informado    3961
Takeaway         3022
In-store         3017
Name: count, dtype: int64


In [37]:
datas_convertidas = pd.to_datetime(
    df_limpo['Transaction Date'],
    format='%Y-%m-%d',
    errors='coerce'
)

df_limpo.loc[
    datas_convertidas.isna(),
    'Transaction Date'
].value_counts(dropna=False)

Transaction Date
NaN        159
UNKNOWN    159
ERROR      142
Name: count, dtype: int64

In [38]:
df_limpo['Transaction Date'] = datas_convertidas

In [39]:
print('Tipo:', df_limpo['Transaction Date'].dtype)
print('Primeira data:', df_limpo['Transaction Date'].min())
print('Última data:', df_limpo['Transaction Date'].max())
print('Sem data:', df_limpo['Transaction Date'].isna().sum())

Tipo: datetime64[us]
Primeira data: 2023-01-01 00:00:00
Última data: 2023-12-31 00:00:00
Sem data: 460


In [40]:
# Preços associados a um único produto na referência observada
produto_por_preco = {
    1.0: 'Cookie',
    1.5: 'Tea',
    2.0: 'Coffee',
    5.0: 'Salad'
}

itens = df_limpo['Item'].astype('string').str.strip()

desconhecidos = (
    itens.isna()
    | itens.str.upper().isin(['UNKNOWN', 'ERROR', '', 'NÃO INFORMADO'])
)

produto_inferido = df_limpo['Price Per Unit'].map(produto_por_preco)
recuperaveis = desconhecidos & produto_inferido.notna()

df_limpo.loc[recuperaveis, 'Item'] = produto_inferido.loc[recuperaveis]
df_limpo.loc[desconhecidos & ~recuperaveis, 'Item'] = 'Não informado'



print('Produtos preenchidos nesta execução:', recuperaveis.sum())
print('Ainda não informados:', df_limpo['Item'].eq('Não informado').sum())

Produtos preenchidos nesta execução: 489
Ainda não informados: 480


In [41]:
df_limpo = df_limpo.drop(
    columns=['Item inferido pelo preço'],
    errors='ignore'
)

In [42]:
print('Dimensões:', df_limpo.shape)
df_limpo.info()

print('Linhas duplicadas:', df_limpo.duplicated().sum())
print('IDs ausentes:', df_limpo['Transaction ID'].isna().sum())
print('IDs repetidos:', df_limpo['Transaction ID'].duplicated().sum())

assert len(df_limpo) == len(df)
assert df_limpo['Transaction ID'].equals(df['Transaction ID'])

Dimensões: (10000, 8)
<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Transaction ID    10000 non-null  str           
 1   Item              10000 non-null  str           
 2   Quantity          9977 non-null   float64       
 3   Price Per Unit    9994 non-null   float64       
 4   Total Spent       9977 non-null   float64       
 5   Payment Method    10000 non-null  str           
 6   Location          10000 non-null  str           
 7   Transaction Date  9540 non-null   datetime64[us]
dtypes: datetime64[us](1), float64(3), str(4)
memory usage: 625.1 KB
Linhas duplicadas: 0
IDs ausentes: 0
IDs repetidos: 0


In [43]:
quantidades = df_limpo['Quantity'].dropna()

assert np.isfinite(quantidades).all()
assert quantidades.gt(0).all()
assert np.isclose(
    quantidades, quantidades.round(), rtol=0, atol=1e-9
).all()

df_limpo['Quantity'] = df_limpo['Quantity'].astype('Int64')

In [44]:
from getpass import getpass

senha = getpass('Senha do PostgreSQL: ')

if not senha:
    raise ValueError('A senha ficou vazia. Execute novamente e preencha o campo.')

print('Senha recebida.')

Senha recebida.


In [45]:
from getpass import getpass
from sqlalchemy import create_engine, URL, text

url = URL.create(
    drivername='postgresql+psycopg',
    username='postgres',
    password=senha,
    host='localhost',
    port=5432,
    database='cafe_sales'
)

engine = create_engine(url)

with engine.connect() as conexao:
    resultado = conexao.execute(
        text('SELECT current_database(), current_user')
    ).one()

print('Banco:', resultado[0])
print('Usuário:', resultado[1])

Banco: cafe_sales
Usuário: postgres


In [46]:
df_carga = df_limpo.rename(columns={
    'Transaction ID': 'transaction_id',
    'Item': 'item',
    'Quantity': 'quantity',
    'Price Per Unit': 'price_per_unit',
    'Total Spent': 'total_spent',
    'Payment Method': 'payment_method',
    'Location': 'location',
    'Transaction Date': 'transaction_date'
}).copy()

print(df_carga.shape)
print(df_carga.columns.tolist())

(10000, 8)
['transaction_id', 'item', 'quantity', 'price_per_unit', 'total_spent', 'payment_method', 'location', 'transaction_date']


In [47]:
with engine.connect() as conexao:
    quantidade = conexao.execute(
        text('SELECT COUNT(*) FROM public.vendas')
    ).scalar_one()

print('Transações já no banco:', quantidade)

Transações já no banco: 0


In [48]:
with engine.begin() as conexao:
    quantidade = conexao.execute(
        text('SELECT COUNT(*) FROM public.vendas')
    ).scalar_one()

    if quantidade != 0:
        raise ValueError('A tabela já contém dados. Carga interrompida.')

    df_carga.to_sql(
        name='vendas',
        con=conexao,
        schema='public',
        if_exists='append',
        index=False,
        chunksize=1000
    )

    total_inserido = conexao.execute(
        text('SELECT COUNT(*) FROM public.vendas')
    ).scalar_one()

    if total_inserido != len(df_carga):
        raise ValueError('A quantidade inserida difere do DataFrame.')

print(f'Carga concluída: {total_inserido} transações.')

Carga concluída: 10000 transações.


In [49]:
print('Unidades vendidas:', df_limpo['Quantity'].sum())
print('Receita conhecida:', round(df_limpo['Total Spent'].sum(), 2))
print('Transações com receita:', df_limpo['Total Spent'].count())
print('Ticket médio:', round(df_limpo['Total Spent'].mean(), 2))

Unidades vendidas: 30180
Receita conhecida: 89096.0
Transações com receita: 9977
Ticket médio: 8.93


In [51]:
with engine.connect() as conexao:
    resultado = conexao.execute(text("""
        SELECT
            current_database() AS banco,
            current_user AS usuario,
            inet_server_addr() AS endereco,
            inet_server_port() AS porta
    """)).mappings().one()

print(dict(resultado))

{'banco': 'cafe_sales', 'usuario': 'postgres', 'endereco': IPv6Address('::1'), 'porta': 5432}
